# Colab + Google Drive bootstrap

Notebook này chỉ kiểm tra runtime, tạo cây thư mục trên Google Drive và ghi báo cáo môi trường. Notebook **không train hoặc export model**. Hãy chọn GPU runtime trước khi chạy, sau đó gửi lại file `colab_environment.json` được in ở cell cuối.

In [ ]:
REPO_URL = "https://github.com/DangHung1110/IoT-CK.git"
GIT_REF = "dev"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/parcel_sorter"  # @param {type:"string"}

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import re
import subprocess
from pathlib import Path

repo_dir = Path("/content/IoT-CK")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(repo_dir)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "--prune"], check=True)
target_ref = GIT_REF if re.fullmatch(r"[0-9a-f]{40}", GIT_REF) else f"origin/{GIT_REF}"
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", target_ref], check=True)

In [ ]:
%pip install -q -r {repo_dir}/requirements/colab.lock.txt
%pip install -q -e {repo_dir} --no-deps

In [ ]:
from parcel_sorter.colab.bootstrap import bootstrap_colab

report_path, report = bootstrap_colab(repo_dir=repo_dir, drive_root=DRIVE_ROOT)
print(report.model_dump_json(indent=2))
print(f"REPORT_PATH={report_path}")